**TRAIN MODEL GIÁ NHÀ HÀ NỘI**

Download các thư viện cẩn thiết

In [1]:
pip install category_encoders xgboost scikit-learn joblib

Mount vào Drive

In [2]:
from google.colab import drive
drive.mount('/content/gdrive', force_remount=True)
%cd '/content/gdrive/MyDrive/Python_Thu5/project'

Mounted at /content/gdrive
/content/gdrive/MyDrive/Python_Thu5/project


Import thư viện

In [3]:

import pandas as pd
import numpy as np
import joblib

from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import mean_absolute_error, r2_score
from category_encoders import TargetEncoder

Load dataset

In [4]:
df = pd.read_csv('housing_cleaned.csv')


Tiền xử lí

In [5]:

# Làm sạch tên cột
df.columns = [col.strip() for col in df.columns]

# Chuẩn hóa Tỉnh / Thành phố
df['Tỉnh/Thành phố'] = df['Tỉnh/Thành phố'].replace({
    'Hà Nội.': 'Hà Nội',
    'Tp Hồ Chí Minh': 'Hồ Chí Minh',
    'TP Hồ Chí Minh': 'Hồ Chí Minh'
}).astype(str).str.strip()

# Fill Quận / Huyện từ Địa chỉ
def fill_geo(row):
    if pd.isnull(row['Quận']) or pd.isnull(row['Huyện']):
        parts = [p.strip() for p in str(row['Địa chỉ']).split(',')]
        if len(parts) >= 2:
            val = parts[-2]
            if pd.isnull(row['Quận']):
                row['Quận'] = val
            if pd.isnull(row['Huyện']):
                row['Huyện'] = val
    return row

df = df.apply(fill_geo, axis=1)

# Điền giá trị thiếu
df['Loại hình nhà ở'] = df['Loại hình nhà ở'].fillna('Không xác định')
df['Giấy tờ pháp lý'] = df['Giấy tờ pháp lý'].fillna('Đang cập nhật')
df['Số tầng'] = df['Số tầng'].fillna(df['Số tầng'].median())
df['Số phòng ngủ'] = df['Số phòng ngủ'].fillna(df['Số phòng ngủ'].median())

# Lọc tỉnh có đủ dữ liệu
city_counts = df['Tỉnh/Thành phố'].value_counts()
df = df[df['Tỉnh/Thành phố'].isin(city_counts[city_counts >= 300].index)]


# =============================
# LẤY RIÊNG HÀ NỘI
# =============================
df = df[df['Tỉnh/Thành phố'] == 'Hà Nội'].copy()

target = 'Giá (triệu đồng/m2)'
df = df.dropna(subset=[target, 'Diện tích'])

# Lọc outliers
df = df[
    (df['Diện tích'].between(15, 500)) &
    (df[target].between(15, 800)) &
    (df['Số tầng'] <= 12)
]

FEATURE ENGINEERING

In [6]:
df['Phường'] = df['Địa chỉ'].apply(
    lambda x: x.split(',')[0].strip() if isinstance(x, str) else 'Unknown'
)
df['Is_Mat_Pho'] = df['Loại hình nhà ở'].apply(
    lambda x: 1 if 'mặt phố' in str(x).lower() else 0
)
df['Total_Floor_Area'] = df['Diện tích'] * df['Số tầng']
df['Quận_LoaiHinh'] = df['Quận'] + "_" + df['Loại hình nhà ở']

features = [
    'Quận', 'Phường', 'Loại hình nhà ở', 'Giấy tờ pháp lý',
    'Số tầng', 'Số phòng ngủ', 'Diện tích',
    'Total_Floor_Area', 'Is_Mat_Pho', 'Quận_LoaiHinh'
]

X = df[features]
y = np.log1p(df[target])

Tiến hành train

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.15, random_state=42
)

cat_cols = ['Quận', 'Phường', 'Loại hình nhà ở', 'Giấy tờ pháp lý', 'Quận_LoaiHinh']
num_cols = ['Số tầng', 'Số phòng ngủ', 'Diện tích', 'Total_Floor_Area']

preprocessor = ColumnTransformer([
    ('cat', TargetEncoder(cols=cat_cols, smoothing=20), cat_cols),
    ('num', RobustScaler(), num_cols),
    ('pass', 'passthrough', ['Is_Mat_Pho'])
])

model = Pipeline([
    ('prep', preprocessor),
    ('xgb', XGBRegressor(
        n_estimators=2000,
        learning_rate=0.02,
        max_depth=9,
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_weight=2,
        random_state=42,
        n_jobs=-1
    ))
])

print("⏳ Training model Hà Nội...")
model.fit(X_train, y_train)

⏳ Training model Hà Nội...


Pipeline(steps=[('prep',
                 ColumnTransformer(transformers=[('cat',
                                                  TargetEncoder(cols=['Quận',
                                                                      'Phường',
                                                                      'Loại '
                                                                      'hình '
                                                                      'nhà ở',
                                                                      'Giấy tờ '
                                                                      'pháp lý',
                                                                      'Quận_LoaiHinh'],
                                                                smoothing=20),
                                                  ['Quận', 'Phường',
                                                   'Loại hình nhà ở',
                                                   'Giấy tờ pháp lý',
                                                   'Quận_LoaiHinh']),
                                                 ('num', RobustScaler(),
                                                  ['Số tầng', 'Số phòng ngủ',
                                                   'Diện tích',
                                                   'Total_Floor_Area']),
                                                 ('pass', 'passthrough',
                                                  ['Is_Mat_Pho'])])),
                (...
                              feature_types=None, feature_weights=None,
                              gamma=None, grow_policy=None,
                              importance_type=None,
                              interaction_constraints=None, learning_rate=0.02,
                              max_bin=None, max_cat_threshold=None,
                              max_cat_to_onehot=None, max_delta_step=None,
                              max_depth=9, max_leaves=None, min_child_weight=2,
                              missing=nan, monotone_constraints=None,
                              multi_strategy=None, n_estimators=2000, n_jobs=-1,
                              num_parallel_tree=None, ...))])

Đánh giá kết quả train dùng R2 và MAE

In [8]:
pred = model.predict(X_test)
print("R2:", r2_score(y_test, pred))
print("MAE:", mean_absolute_error(np.expm1(y_test), np.expm1(pred)))

R2: 0.594609408758773
MAE: 17.772534064477583


Lưu model để sử dụng

In [11]:
joblib.dump(model, 'hanoi_house_model.pkl')
print("Saved hanoi_house_model.pkl")

Saved hanoi_house_model.pkl
